In [ ]:
import os
import json
import ollama


JSON_PATH = "Transcripciones\\out-3851-5836-20260504-102528-1777908328.6529351.json"


def build_transcription_from_json(path):
    if not os.path.exists(path):
        return None
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    segments = data.get("segments", [])
    parts = []
    for seg in segments:
        speaker = seg.get("speaker") or seg.get("speaker_label") or "SPEAKER"
        text = seg.get("text", "").strip()
        if text:
            # normalize whitespace
            text = " ".join(text.split())
            parts.append(f"{speaker}: {text}")

    return "\n".join(parts)


transcripcion = build_transcription_from_json(JSON_PATH)
if not transcripcion:
    # fallback example if JSON not found
    transcripcion = """
AGENTE: Buenas tardes, en qué puedo ayudarle.
CLIENTE: Estoy molesto porque me cobraron doble este mes.
AGENTE: Entiendo su molestia, voy a revisar su caso.
CLIENTE: Gracias.
"""


prompt = f"""
Analiza la siguiente llamada de contact center.

Devuelve el resultado en formato JSON con las claves:

"empatia_agente": numero_0_100,
"frustracion_cliente": numero_0_100,
"resolucion": numero_0_100,
"tipo_llamada": "string",
"resumen": "string"

Llamada:
{transcripcion}
"""

response = ollama.chat(
    model='llama3.1:8b',
    messages=[
        {
            'role': 'user',
            'content': prompt
        }
    ]
)

print(response['message']['content'])

# guarad la respuesta del modelo en un archivo JSON
with open("analisis_llamada2.json", "w", encoding="utf-8") as f:
    json.dump(response['message']['content'], f, ensure_ascii=False, indent=4)  

Análisis de la llamada de contacto:

### Empatia Agente (0.6/5)

*   La empatía del agente es baja debido a que se muestra frustrado por las preguntas repetitivas y los malentendidos.
*   El agente utiliza un lenguaje áspero y sarcástico con el cliente, lo cual no ayuda a resolver la situación.

### Frustración Cliente (4.8/5)

*   La frustración del cliente es alta debido a que no logra entender cómo funciona la promoción de los productos.
*   El cliente se muestra impaciente y molesto por las respuestas del agente, lo cual hace que la situación empeore.

### Resolución (2.4/5)

*   La resolución de la llamada es baja debido a que el agente no logra resolver completamente el problema del cliente.
*   El agente se enfoca en pasar al cliente a otra persona, lo cual puede ser visto como una evasiva y no un intento de resolver la situación.

### Tipo Llamada (1.3/5)

*   La llamada es clasificada como "problemática" debido a que el cliente tiene problemas con la promoción de los productos

In [5]:
transcripcion

'SPEAKER: Muy buenos días, Formación Medigitierra de Gabriela Saludan.\nSPEAKER: ¿Qué le podemos ayudar?\nSPEAKER: Como está estimada, muy buenos días.\nSPEAKER: Les saluda Rona Eras, Asesor Operativo.\nSPEAKER: ¿En general un ticket por soporte de sistemas a nombre de Janet Boracá?\nSPEAKER: Sí, la misma.\nSPEAKER: Con vita le voy a molestar.\nSPEAKER: Lo que pasa es que aquí tengo a una cliente que hizo una compra de una medicina contínua.\nSPEAKER: Compró dos productos y le salió uno gratis.\nSPEAKER: Pero ahorita la facturación que le realicé en la factura no le salió gratis.\nSPEAKER: Sino que se fue automáticamente directo al cobro.\nSPEAKER: Ya, permítame por favor en este momento revisar el número de cédula de la cliente que me ayude, por favor.\nSPEAKER: Ya, es el número diecisiete.\nSPEAKER: Un segundito para ingresar al sistema, ¿Sí?\nSPEAKER: Ya, ¿Me puede indicar qué producto es y el número de cédula del cliente?\nSPEAKER: Ya, número de cédula diecisiete, doce, veinte, cin

In [10]:
pip install reportlab

   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   --------------------- ------------------ 1.0/2.0 MB 4.6 MB/s eta 0:00:01
   ---------------------------------------- 2.0/2.0 MB 5.2 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os
import json
from collections import Counter

import ollama
import pandas as pd
import matplotlib.pyplot as plt

from reportlab.lib import colors
from reportlab.lib.pagesizes import letter
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.enums import TA_CENTER

from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer,
    Table,
    TableStyle,
    Image,
    PageBreak
)

from reportlab.lib.styles import ParagraphStyle
from reportlab.lib.units import inch


# =========================================================
# CONFIGURACIÓN
# =========================================================

CARPETA_TRANSCRIPCIONES = "Transcripciones_diarización"

CARPETA_ANALISIS = "Analisis_diarización"

PDF_SALIDA = "Reporte_Agente.pdf"

AGENTE_NOMBRE = "Juan Pérez"

MODELO_OLLAMA = "llama3.1:8b"

os.makedirs(CARPETA_ANALISIS, exist_ok=True)


# =========================================================
# FUNCIONES
# =========================================================

def build_transcription_from_json(path):

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    segments = data.get("segments", [])

    parts = []

    for seg in segments:

        speaker = (
            seg.get("speaker")
            or seg.get("speaker_label")
            or "SPEAKER"
        )

        text = seg.get("text", "").strip()

        if text:

            text = " ".join(text.split())

            parts.append(f"{speaker}: {text}")

    return "\n".join(parts)


def analizar_llamada(transcripcion):

    prompt = f"""
Analiza la siguiente llamada de contact center.

Responde ÚNICAMENTE un JSON válido.

Formato exacto:

{{
  "empatia_agente": numero_0_100,
  "frustracion_cliente": numero_0_100,
  "resolucion": numero_0_100,
  "tipo_llamada": "string",
  "resumen": "string"
}}

Reglas:
- empatia_agente: nivel de empatía del asesor
- frustracion_cliente: nivel de frustración detectado
- resolucion: capacidad de resolución del agente
- tipo_llamada: reclamo, consulta, soporte, facturación, etc.
- resumen: máximo 40 palabras
- devolver SOLO JSON
- no agregar explicaciones

Llamada:
{transcripcion}
"""

    response = ollama.chat(
        model=MODELO_OLLAMA,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    contenido = response["message"]["content"]

    contenido = contenido.strip()

    inicio = contenido.find("{")
    fin = contenido.rfind("}") + 1

    contenido = contenido[inicio:fin]

    return json.loads(contenido)


def generar_resumen_global(df):

    resumen_llamadas = "\n".join(
        [
            f"- {r['tipo_llamada']}: {r['resumen']}"
            for _, r in df.iterrows()
        ]
    )

    prompt = f"""
Analiza el desempeño general del asesor.

Genera un resumen ejecutivo profesional de máximo 120 palabras.

Incluye:
- empatía
- resolución
- frustración de clientes
- problemas recurrentes

Información:
{resumen_llamadas}
"""

    response = ollama.chat(
        model=MODELO_OLLAMA,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response["message"]["content"]


def crear_piechart(df):

    conteo = df["tipo_llamada"].value_counts()

    plt.figure(figsize=(5, 5))

    plt.pie(
        conteo.values,
        labels=conteo.index,
        autopct='%1.1f%%'
    )

    plt.title("Distribución Tipo de Llamadas")

    ruta = "piechart.png"

    plt.savefig(
        ruta,
        bbox_inches="tight"
    )

    plt.close()

    return ruta


# =========================================================
# PROCESAR LLAMADAS
# =========================================================

resultados = []

archivos = [
    f for f in os.listdir(CARPETA_TRANSCRIPCIONES)
    if f.endswith(".json")
]

print(f"JSON encontrados: {len(archivos)}")

for archivo in archivos:

    try:

        print(f"Procesando: {archivo}")

        ruta = os.path.join(
            CARPETA_TRANSCRIPCIONES,
            archivo
        )

        transcripcion = build_transcription_from_json(ruta)

        analisis = analizar_llamada(transcripcion)

        analisis["archivo"] = archivo

        resultados.append(analisis)

        # guardar análisis individual

        ruta_analisis = os.path.join(
            CARPETA_ANALISIS,
            archivo
        )

        with open(
            ruta_analisis,
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                analisis,
                f,
                ensure_ascii=False,
                indent=4
            )

    except Exception as e:

        print(f"Error procesando {archivo}")
        print(e)


# =========================================================
# DATAFRAME
# =========================================================

df = pd.DataFrame(resultados)

if df.empty:

    print("No existen resultados.")
    exit()


# =========================================================
# KPIs
# =========================================================

empatia_prom = round(
    df["empatia_agente"].mean(),
    1
)

frustracion_prom = round(
    df["frustracion_cliente"].mean(),
    1
)

resolucion_prom = round(
    df["resolucion"].mean(),
    1
)

score_global = round(
    (
        empatia_prom * 0.50
        + resolucion_prom * 0.45
        - frustracion_prom * 0.05
    ),
    1
)


# =========================================================
# RESUMEN EJECUTIVO
# =========================================================

resumen_global = generar_resumen_global(df)


# =========================================================
# PIECHART
# =========================================================

ruta_piechart = crear_piechart(df)


# =========================================================
# PDF
# =========================================================

doc = SimpleDocTemplate(
    PDF_SALIDA,
    pagesize=letter,
    rightMargin=40,
    leftMargin=40,
    topMargin=40,
    bottomMargin=40
)

styles = getSampleStyleSheet()

elements = []


# =========================================================
# ESTILOS
# =========================================================

titulo_style = ParagraphStyle(
    'Titulo',
    parent=styles['Heading1'],
    alignment=TA_CENTER,
    fontSize=22,
    textColor=colors.HexColor("#1F3A5F"),
    spaceAfter=20
)

subtitulo_style = ParagraphStyle(
    'Subtitulo',
    parent=styles['Heading2'],
    textColor=colors.HexColor("#1F3A5F"),
    spaceAfter=12
)

normal_style = styles["BodyText"]


# =========================================================
# PORTADA
# =========================================================

elements.append(
    Paragraph(
        "Reporte Inteligente de Calidad de Atención",
        titulo_style
    )
)

elements.append(
    Paragraph(
        f"<b>Agente:</b> {AGENTE_NOMBRE}",
        normal_style
    )
)

elements.append(
    Paragraph(
        f"<b>Total llamadas analizadas:</b> {len(df)}",
        normal_style
    )
)

elements.append(
    Paragraph(
        f"<b>Modelo IA:</b> WhisperX + Llama 3.1",
        normal_style
    )
)

elements.append(Spacer(1, 25))


# =========================================================
# KPIS
# =========================================================

elements.append(
    Paragraph(
        "KPIs Principales",
        subtitulo_style
    )
)

kpi_data = [

    [
        "Empatía Agente",
        "Frustración Cliente",
        "Resolución",
        "Score Global"
    ],

    [
        f"{empatia_prom}/100",
        f"{frustracion_prom}/100",
        f"{resolucion_prom}/100",
        f"{score_global}/100"
    ]
]

kpi_table = Table(
    kpi_data,
    colWidths=[120, 120, 120, 120]
)

kpi_table.setStyle(TableStyle([

    ('BACKGROUND', (0, 0), (-1, 0), colors.HexColor("#1F3A5F")),

    ('TEXTCOLOR', (0, 0), (-1, 0), colors.white),

    ('ALIGN', (0, 0), (-1, -1), 'CENTER'),

    ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),

    ('FONTSIZE', (0, 1), (-1, 1), 18),

    ('BOTTOMPADDING', (0, 0), (-1, 0), 12),

    ('BACKGROUND', (0, 1), (-1, 1), colors.HexColor("#EAF2F8")),

    ('GRID', (0, 0), (-1, -1), 1, colors.grey),
    
    ('VALIGN', (0, 0), (-1, -1), 'MIDDLE')

]))

elements.append(kpi_table)

elements.append(Spacer(1, 30))


# =========================================================
# RESUMEN EJECUTIVO
# =========================================================

elements.append(
    Paragraph(
        "Resumen Ejecutivo",
        subtitulo_style
    )
)

elements.append(
    Paragraph(
        resumen_global,
        normal_style
    )
)

elements.append(Spacer(1, 25))


# =========================================================
# PIE CHART
# =========================================================

elements.append(
    Paragraph(
        "Distribución de Tipo de Llamadas",
        subtitulo_style
    )
)

img = Image(
    ruta_piechart,
    width=4.5 * inch,
    height=4.5 * inch
)

elements.append(img)

elements.append(Spacer(1, 25))


# =========================================================
# DETALLE LLAMADA A LLAMADA
# =========================================================

elements.append(
    Paragraph(
        "Detalle Llamada a Llamada",
        subtitulo_style
    )
)

tabla_detalle = [[

    "Archivo",
    "Tipo",
    "Empatía",
    "Frustración",
    "Resolución",
    "Resumen"
]]

for _, row in df.iterrows():
    
    nombre_corto = row.get("archivo", "")[:11]
    
    resumen_paragraph = Paragraph(
        row.get("resumen", ""),
        normal_style
    )
    
    tipo_paragraph = Paragraph(
        row.get("tipo_llamada", ""),
        normal_style
    )

    tabla_detalle.append([

        nombre_corto,

        tipo_paragraph,

        str(row["empatia_agente"]),

        str(row["frustracion_cliente"]),

        str(row["resolucion"]),

        resumen_paragraph
    ])

detalle_table = Table(
    tabla_detalle,
    colWidths=[90, 70, 60, 70, 60, 170]
)

detalle_table.setStyle(TableStyle([

    ('BACKGROUND', (0, 0), (-1, 0), colors.HexColor("#1F3A5F")),

    ('TEXTCOLOR', (0, 0), (-1, 0), colors.white),

    ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),

    ('FONTSIZE', (0, 0), (-1, 0), 10),

    ('GRID', (0, 0), (-1, -1), 0.5, colors.grey),

    ('BACKGROUND', (0, 1), (-1, -1), colors.whitesmoke),

    ('VALIGN', (0, 0), (-1, -1), 'TOP'),

    ('FONTSIZE', (0, 1), (-1, -1), 8)

]))

elements.append(detalle_table)


# =========================================================
# GENERAR PDF
# =========================================================

doc.build(elements)

print("\n===================================")
print("PDF generado correctamente")
print(PDF_SALIDA)

JSON encontrados: 30
Procesando: out-0960926727-7100-20260504-111501-1777911301.6532715.json
Error procesando out-0960926727-7100-20260504-111501-1777911301.6532715.json
Expecting value: line 1 column 1 (char 0)
Procesando: out-0968007375-5671-20260504-121227-1777914747.6536155.json
Procesando: out-0984990407-5836-20260504-105047-1777909847.6531071.json
Procesando: out-0987763848-5828-20260504-104423-1777909463.6530636.json
Procesando: out-0989253909-5991-20260504-103031-1777908631.6529648.json
Procesando: out-0990334974-5611-20260504-124338-1777916618.6537667.json
Procesando: out-0993660680-5830-20260504-104528-1777909528.6530687.json
Procesando: out-0997155966-5810-20260504-123305-1777915985.6537096.json
Procesando: out-2209-5991-20260504-080849-1777900129.6522391.json
Procesando: out-2340-5671-20260504-100723-1777907243.6528238.json
Procesando: out-2520-5725-20260504-124711-1777916831.6537870.json
Procesando: out-2524-5671-20260504-090550-1777903550.6524525.json
Error procesando out

In [ ]:
import os
import json


import ollama
import pandas as pd
import matplotlib.pyplot as plt
from reportlab.platypus import Paragraph

from reportlab.lib import colors
from reportlab.lib.pagesizes import letter
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.enums import TA_CENTER

from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer,
    Table,
    TableStyle,
    Image
)

from reportlab.lib.styles import ParagraphStyle
from reportlab.lib.units import inch



# =========================================================
# FUNCIONES
# =========================================================


def generar_resumen_global(df):

    resumen_llamadas = "\n".join(
        [
            f"- {r['tipo_llamada']}: {r['resumen']}"
            for _, r in df.iterrows()
        ]
    )

    prompt = f"""
Analiza el desempeño general del asesor.

Genera un resumen ejecutivo profesional de máximo 120 palabras.

Incluye:
- empatía
- resolución
- frustración de clientes
- problemas recurrentes

Información:
{resumen_llamadas}
"""

    response = ollama.chat(
        model=MODELO_OLLAMA,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response["message"]["content"]





# =========================================================
# CONFIGURACIÓN
# =========================================================

CARPETA_ANALISIS = "Analisis"

PDF_SALIDA = "Reporte_Agente_v2.pdf"

AGENTE_NOMBRE = "Juan Pérez"


# =========================================================
# CARGAR JSONS ANALISIS
# =========================================================

resultados = []

archivos = [
    f for f in os.listdir(CARPETA_ANALISIS)
    if f.endswith(".json")
]

print(f"Archivos encontrados: {len(archivos)}")


for archivo in archivos:

    try:

        ruta = os.path.join(
            CARPETA_ANALISIS,
            archivo
        )

        with open(
            ruta,
            "r",
            encoding="utf-8"
        ) as f:

            data = json.load(f)

        data["archivo"] = archivo

        resultados.append(data)

    except Exception as e:

        print(f"Error leyendo {archivo}")
        print(e)


# =========================================================
# DATAFRAME
# =========================================================

df = pd.DataFrame(resultados)

if df.empty:

    print("No existen datos.")
    exit()


# =========================================================
# LIMPIEZA NUMÉRICA
# =========================================================

columnas_numericas = [

    "empatia_agente",
    "frustracion_cliente",
    "resolucion"
]

for col in columnas_numericas:

    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

df = df.dropna(subset=columnas_numericas)


# =========================================================
# KPIs
# =========================================================

empatia_prom = round(
    df["empatia_agente"].mean(),
    1
)

frustracion_prom = round(
    df["frustracion_cliente"].mean(),
    1
)

resolucion_prom = round(
    df["resolucion"].mean(),
    1
)

score_global = round(
    (
        empatia_prom * 0.50
        + resolucion_prom * 0.45
        - frustracion_prom * 0.05
    ),
    1
)


# =========================================================
# RESUMEN EJECUTIVO SIMPLE
# =========================================================

tipo_frecuente = (
    df["tipo_llamada"]
    .value_counts()
    .idxmax()
)

resumen_global = f"""
El agente analizado presenta un nivel promedio de empatía de {empatia_prom}/100
y una capacidad de resolución de {resolucion_prom}/100.

El nivel promedio de frustración detectado en clientes fue de
{frustracion_prom}/100.

El tipo de llamada más frecuente corresponde a:
<b>{tipo_frecuente}</b>.

El score global de desempeño calculado por IA fue de
<b>{score_global}/100</b>.
"""


# =========================================================
# RESUMEN EJECUTIVO IA
# =========================================================

resumen_global = generar_resumen_global(df)

# =========================================================
# PIE CHART
# =========================================================

conteo = df["tipo_llamada"].value_counts()

plt.figure(figsize=(5, 5))

plt.pie(
    conteo.values,
    labels=conteo.index,
    autopct='%1.1f%%'
)

plt.title("Distribución Tipo de Llamadas")

ruta_piechart = "piechart.png"

plt.savefig(
    ruta_piechart,
    bbox_inches="tight"
)

plt.close()


# =========================================================
# PDF
# =========================================================

doc = SimpleDocTemplate(
    PDF_SALIDA,
    pagesize=letter,
    rightMargin=40,
    leftMargin=40,
    topMargin=40,
    bottomMargin=40
)

styles = getSampleStyleSheet()

elements = []


# =========================================================
# ESTILOS
# =========================================================

titulo_style = ParagraphStyle(
    'Titulo',
    parent=styles['Heading1'],
    alignment=TA_CENTER,
    fontSize=22,
    textColor=colors.HexColor("#1F3A5F"),
    spaceAfter=20
)

subtitulo_style = ParagraphStyle(
    'Subtitulo',
    parent=styles['Heading2'],
    textColor=colors.HexColor("#1F3A5F"),
    spaceAfter=12
)

normal_style = styles["BodyText"]


# =========================================================
# PORTADA
# =========================================================

elements.append(
    Paragraph(
        "Reporte Inteligente de Calidad de Atención",
        titulo_style
    )
)

elements.append(
    Paragraph(
        f"<b>Agente:</b> {AGENTE_NOMBRE}",
        normal_style
    )
)

elements.append(
    Paragraph(
        f"<b>Total llamadas analizadas:</b> {len(df)}",
        normal_style
    )
)

elements.append(
    Paragraph(
        "<b>Fuente:</b> Archivos JSON de análisis IA",
        normal_style
    )
)

elements.append(Spacer(1, 25))


# =========================================================
# KPIS
# =========================================================

elements.append(
    Paragraph(
        "KPIs Principales",
        subtitulo_style
    )
)

kpi_data = [

    [
        "Empatía Agente",
        "Frustración Cliente",
        "Resolución",
        "Score Global"
    ],

    [
        f"{empatia_prom}/100",
        f"{frustracion_prom}/100",
        f"{resolucion_prom}/100",
        f"{score_global}/100"
    ]
]

kpi_table = Table(
    kpi_data,
    colWidths=[120, 120, 120, 120],
    rowHeights=[35, 55]
)

kpi_table.setStyle(TableStyle([

    ('BACKGROUND', (0, 0), (-1, 0), colors.HexColor("#1F3A5F")),

    ('TEXTCOLOR', (0, 0), (-1, 0), colors.white),

    ('ALIGN', (0, 0), (-1, -1), 'CENTER'),

    ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),

    ('FONTSIZE', (0, 1), (-1, 1), 18),

    ('BOTTOMPADDING', (0, 0), (-1, 0), 12),

    ('BACKGROUND', (0, 1), (-1, 1), colors.HexColor("#EAF2F8")),

    ('GRID', (0, 0), (-1, -1), 1, colors.grey),
    
    ('VALIGN', (0, 0), (-1, -1), 'MIDDLE'),

]))

elements.append(kpi_table)

elements.append(Spacer(1, 30))


# =========================================================
# RESUMEN EJECUTIVO
# =========================================================

elements.append(
    Paragraph(
        "Resumen Ejecutivo",
        subtitulo_style
    )
)

elements.append(
    Paragraph(
        resumen_global,
        normal_style
    )
)

elements.append(Spacer(1, 25))


# =========================================================
# PIE CHART
# =========================================================

elements.append(
    Paragraph(
        "Distribución de Tipo de Llamadas",
        subtitulo_style
    )
)

img = Image(
    ruta_piechart,
    width=4.5 * inch,
    height=4.5 * inch
)

elements.append(img)

elements.append(Spacer(1, 25))


# =========================================================
# DETALLE LLAMADA A LLAMADA
# =========================================================

elements.append(
    Paragraph(
        "Detalle Llamada a Llamada",
        subtitulo_style
    )
)

tabla_detalle = [[

    "Archivo",
    "Tipo",
    "Empatía",
    "Frustración",
    "Resolución",
    "Resumen"
]]

for _, row in df.iterrows():
    nombre_corto = row.get("archivo", "")[:11]
    
    resumen_paragraph = Paragraph(
        row.get("resumen", ""),
        normal_style
    )
    
    tipo_paragraph = Paragraph(
        row.get("tipo_llamada", ""),
        normal_style
    )

    tabla_detalle.append([

        nombre_corto,

        tipo_paragraph,

        str(row.get("empatia_agente", "")),

        str(row.get("frustracion_cliente", "")),

        str(row.get("resolucion", "")),

        resumen_paragraph
    ])

detalle_table = Table(
    tabla_detalle,
    colWidths=[90, 70, 60, 70, 60, 170]
)

detalle_table.setStyle(TableStyle([

    ('BACKGROUND', (0, 0), (-1, 0), colors.HexColor("#1F3A5F")),

    ('TEXTCOLOR', (0, 0), (-1, 0), colors.white),

    ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),

    ('FONTSIZE', (0, 0), (-1, 0), 10),

    ('GRID', (0, 0), (-1, -1), 0.5, colors.grey),

    ('BACKGROUND', (0, 1), (-1, -1), colors.whitesmoke),

    ('VALIGN', (0, 0), (-1, -1), 'TOP'),

    ('FONTSIZE', (0, 1), (-1, -1), 8)

]))

elements.append(detalle_table)


# =========================================================
# GENERAR PDF
# =========================================================

doc.build(elements)

print("\n===================================")
print("PDF generado correctamente")
print(PDF_SALIDA)

Archivos encontrados: 27

PDF generado correctamente
Reporte_Agente_v2.pdf


In [ ]:
# =========================================================
# CONFIGURACIÓN NUEVA
# =========================================================

CARPETA_ANALISIS = "Analisis"

CARPETA_SENTIMIENTOS = "Sentimientos"

PDF_SALIDA = "Reporte_Agente_v3.pdf"

AGENTE_NOMBRE = "Juan Pérez"


# =========================================================
# CARGAR JSONS ANALISIS CONTEXTUAL
# =========================================================

resultados = []

archivos = [
    f for f in os.listdir(CARPETA_ANALISIS)
    if f.endswith(".json")
]

print(f"Archivos análisis encontrados: {len(archivos)}")


for archivo in archivos:

    try:

        ruta = os.path.join(
            CARPETA_ANALISIS,
            archivo
        )

        with open(
            ruta,
            "r",
            encoding="utf-8"
        ) as f:

            data = json.load(f)

        data["archivo"] = archivo

        resultados.append(data)

    except Exception as e:

        print(f"Error leyendo análisis {archivo}")
        print(e)


# =========================================================
# DATAFRAME ANALISIS
# =========================================================

df = pd.DataFrame(resultados)

if df.empty:

    print("No existen datos.")
    exit()


# =========================================================
# CARGAR JSONS SENTIMIENTOS
# =========================================================

resultados_sentimientos = []

archivos_sentimientos = [

    f for f in os.listdir(CARPETA_SENTIMIENTOS)
    if f.endswith(".json")
]

print(f"Archivos sentimientos encontrados: {len(archivos_sentimientos)}")


for archivo in archivos_sentimientos:

    try:

        ruta = os.path.join(
            CARPETA_SENTIMIENTOS,
            archivo
        )

        with open(
            ruta,
            "r",
            encoding="utf-8"
        ) as f:

            data = json.load(f)

        resultados_sentimientos.append(data)

    except Exception as e:

        print(f"Error leyendo sentimiento {archivo}")
        print(e)


# =========================================================
# DATAFRAME SENTIMIENTOS
# =========================================================

df_sent = pd.DataFrame(resultados_sentimientos)

if df_sent.empty:

    print("No existen datos de sentimientos.")
    exit()


# =========================================================
# MERGE
# =========================================================

df_total = df.merge(
    df_sent,
    on="archivo",
    how="left"
)


# =========================================================
# LIMPIEZA NUMÉRICA
# =========================================================

columnas_numericas = [

    "empatia_agente",
    "frustracion_cliente",
    "resolucion",
    "score_pos",
    "score_neu",
    "score_neg"
]

for col in columnas_numericas:

    df_total[col] = pd.to_numeric(
        df_total[col],
        errors="coerce"
    )

df_total = df_total.dropna(
    subset=columnas_numericas
)


# =========================================================
# KPIs PRINCIPALES
# =========================================================

empatia_prom = round(
    df_total["empatia_agente"].mean(),
    1
)

frustracion_prom = round(
    df_total["frustracion_cliente"].mean(),
    1
)

resolucion_prom = round(
    df_total["resolucion"].mean(),
    1
)

score_global = round(
    (
        empatia_prom * 0.50
        + resolucion_prom * 0.45
        - frustracion_prom * 0.05
    ),
    1
)


# =========================================================
# KPIs SENTIMIENTO
# =========================================================

pos_pct = round(
    (
        (df_total["sentimiento"] == "POS").sum()
        / len(df_total)
    ) * 100,
    1
)

neu_pct = round(
    (
        (df_total["sentimiento"] == "NEU").sum()
        / len(df_total)
    ) * 100,
    1
)

neg_pct = round(
    (
        (df_total["sentimiento"] == "NEG").sum()
        / len(df_total)
    ) * 100,
    1
)


# =========================================================
# RESUMEN EJECUTIVO IA
# =========================================================

resumen_global = generar_resumen_global(df_total)


# =========================================================
# PIE CHART
# =========================================================

conteo = df_total["tipo_llamada"].value_counts()

plt.figure(figsize=(5, 5))

plt.pie(
    conteo.values,
    labels=conteo.index,
    autopct='%1.1f%%'
)

plt.title("Distribución Tipo de Llamadas")

ruta_piechart = "piechart.png"

plt.savefig(
    ruta_piechart,
    bbox_inches="tight"
)

plt.close()


# =========================================================
# PDF
# =========================================================

doc = SimpleDocTemplate(
    PDF_SALIDA,
    pagesize=letter,
    rightMargin=40,
    leftMargin=40,
    topMargin=40,
    bottomMargin=40
)

styles = getSampleStyleSheet()

elements = []


# =========================================================
# ESTILOS
# =========================================================

titulo_style = ParagraphStyle(
    'Titulo',
    parent=styles['Heading1'],
    alignment=TA_CENTER,
    fontSize=22,
    textColor=colors.HexColor("#1F3A5F"),
    spaceAfter=20
)

subtitulo_style = ParagraphStyle(
    'Subtitulo',
    parent=styles['Heading2'],
    textColor=colors.HexColor("#1F3A5F"),
    spaceAfter=12
)

normal_style = styles["BodyText"]


# =========================================================
# PORTADA
# =========================================================

elements.append(
    Paragraph(
        "Reporte Inteligente de Calidad de Atención",
        titulo_style
    )
)

elements.append(
    Paragraph(
        f"<b>Agente:</b> {AGENTE_NOMBRE}",
        normal_style
    )
)

elements.append(
    Paragraph(
        f"<b>Total llamadas analizadas:</b> {len(df_total)}",
        normal_style
    )
)

elements.append(Spacer(1, 25))


# =========================================================
# KPIS PRINCIPALES
# =========================================================

elements.append(
    Paragraph(
        "KPIs Principales",
        subtitulo_style
    )
)

kpi_data = [

    [
        "Empatía",
        "Frustración",
        "Resolución",
        "Score Global"
    ],

    [
        f"{empatia_prom}/100",
        f"{frustracion_prom}/100",
        f"{resolucion_prom}/100",
        f"{score_global}/100"
    ]
]

kpi_table = Table(
    kpi_data,
    colWidths=[120, 120, 120, 120],
    rowHeights=[35, 55]
)

kpi_table.setStyle(TableStyle([

    ('BACKGROUND', (0, 0), (-1, 0), colors.HexColor("#1F3A5F")),

    ('TEXTCOLOR', (0, 0), (-1, 0), colors.white),

    ('ALIGN', (0, 0), (-1, -1), 'CENTER'),

    ('VALIGN', (0, 0), (-1, -1), 'MIDDLE'),

    ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),

    ('FONTSIZE', (0, 1), (-1, 1), 18),

    ('BACKGROUND', (0, 1), (-1, 1), colors.HexColor("#EAF2F8")),

    ('GRID', (0, 0), (-1, -1), 1, colors.grey)

]))

elements.append(kpi_table)

elements.append(Spacer(1, 20))


# =========================================================
# KPIS SENTIMIENTOS
# =========================================================

elements.append(
    Paragraph(
        "Distribución de Polaridades",
        subtitulo_style
    )
)

sent_data = [

    [
        "Positivo",
        "Neutral",
        "Negativo"
    ],

    [
        f"{pos_pct}%",
        f"{neu_pct}%",
        f"{neg_pct}%"
    ]
]

sent_table = Table(
    sent_data,
    colWidths=[160, 160, 160],
    rowHeights=[35, 50]
)

sent_table.setStyle(TableStyle([

    ('BACKGROUND', (0, 0), (-1, 0), colors.HexColor("#145A32")),

    ('TEXTCOLOR', (0, 0), (-1, 0), colors.white),

    ('ALIGN', (0, 0), (-1, -1), 'CENTER'),

    ('VALIGN', (0, 0), (-1, -1), 'MIDDLE'),

    ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),

    ('FONTSIZE', (0, 1), (-1, 1), 16),

    ('BACKGROUND', (0, 1), (-1, 1), colors.HexColor("#EAFAF1")),

    ('GRID', (0, 0), (-1, -1), 1, colors.grey)

]))

elements.append(sent_table)

elements.append(Spacer(1, 30))


# =========================================================
# RESUMEN EJECUTIVO
# =========================================================

elements.append(
    Paragraph(
        "Resumen Ejecutivo",
        subtitulo_style
    )
)

elements.append(
    Paragraph(
        resumen_global,
        normal_style
    )
)

elements.append(Spacer(1, 25))


# =========================================================
# PIE CHART
# =========================================================

elements.append(
    Paragraph(
        "Distribución de Tipo de Llamadas",
        subtitulo_style
    )
)

img = Image(
    ruta_piechart,
    width=4.5 * inch,
    height=4.5 * inch
)

elements.append(img)

elements.append(Spacer(1, 25))


# =========================================================
# DETALLE LLAMADAS
# =========================================================

elements.append(
    Paragraph(
        "Detalle Llamada a Llamada",
        subtitulo_style
    )
)

tabla_detalle = [[

    "Archivo",
    "Tipo",
    "Empatía",
    "Frustración",
    "Resolución",
    "Resumen"
]]

for _, row in df_total.iterrows():

    nombre_corto = row.get("archivo", "")[:11]

    resumen_paragraph = Paragraph(
        row.get("resumen", ""),
        normal_style
    )

    tipo_paragraph = Paragraph(
        row.get("tipo_llamada", ""),
        normal_style
    )

    tabla_detalle.append([

        nombre_corto,

        tipo_paragraph,

        str(row.get("empatia_agente", "")),

        str(row.get("frustracion_cliente", "")),

        str(row.get("resolucion", "")),

        resumen_paragraph
    ])

detalle_table = Table(
    tabla_detalle,
    colWidths=[65, 70, 55, 65, 55, 230]
)

detalle_table.setStyle(TableStyle([

    ('BACKGROUND', (0, 0), (-1, 0), colors.HexColor("#1F3A5F")),

    ('TEXTCOLOR', (0, 0), (-1, 0), colors.white),

    ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),

    ('FONTSIZE', (0, 0), (-1, 0), 10),

    ('GRID', (0, 0), (-1, -1), 0.5, colors.grey),

    ('BACKGROUND', (0, 1), (-1, -1), colors.whitesmoke),

    ('VALIGN', (0, 0), (-1, -1), 'TOP'),

    ('FONTSIZE', (0, 1), (-1, -1), 7)

]))

elements.append(detalle_table)

elements.append(Spacer(1, 30))


# =========================================================
# TABLA SENTIMIENTOS
# =========================================================

elements.append(
    Paragraph(
        "Detalle de Polaridades por Llamada",
        subtitulo_style
    )
)

tabla_sentimientos = [[

    "Archivo",
    "Polaridad",
    "POS",
    "NEU",
    "NEG"
]]

for _, row in df_total.iterrows():

    nombre_corto = row.get("archivo", "")[:11]

    tabla_sentimientos.append([

        nombre_corto,

        row.get("sentimiento", ""),

        str(round(row.get("score_pos", 0), 3)),

        str(round(row.get("score_neu", 0), 3)),

        str(round(row.get("score_neg", 0), 3))
    ])

tabla_sent = Table(
    tabla_sentimientos,
    colWidths=[120, 100, 80, 80, 80]
)

tabla_sent.setStyle(TableStyle([

    ('BACKGROUND', (0, 0), (-1, 0), colors.HexColor("#7D6608")),

    ('TEXTCOLOR', (0, 0), (-1, 0), colors.white),

    ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),

    ('ALIGN', (0, 0), (-1, -1), 'CENTER'),

    ('GRID', (0, 0), (-1, -1), 0.5, colors.grey),

    ('BACKGROUND', (0, 1), (-1, -1), colors.HexColor("#FCF3CF")),

    ('FONTSIZE', (0, 1), (-1, -1), 8)

]))

elements.append(tabla_sent)


# =========================================================
# GENERAR PDF
# =========================================================

doc.build(elements)

print("\n===================================")
print("PDF generado correctamente")
print(PDF_SALIDA)

Archivos análisis encontrados: 27
Archivos sentimientos encontrados: 28

PDF generado correctamente
Reporte_Agente_v3.pdf
